## Imports and Loading

In [386]:
import datetime
import numpy as np
import pandas as pd
from pprint import pprint

In [387]:
df = pd.read_csv("data/raw.csv")

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

Loaded dataset: 1020 rows, 12 columns


In [388]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [389]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   str    
 1   First_Name         1020 non-null   str    
 2   Last_Name          1020 non-null   str    
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   str    
 5   Status             1020 non-null   str    
 6   Join_Date          1020 non-null   str    
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   str    
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   str    
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), str(8)
memory usage: 88.8 KB


In [390]:
df.isna().sum()

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

## Whitespace and Duplicate Cleaning

In [391]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print(df.columns.tolist())

['employee_id', 'first_name', 'last_name', 'age', 'department_region', 'status', 'join_date', 'salary', 'email', 'phone', 'performance_score', 'remote_work']


In [392]:
string_cols = df.select_dtypes(include=["object", "string"]).columns

for col in string_cols:
    df[col] = df[col].str.strip()

In [393]:
print("Number of duplicated records/rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)

Number of duplicated records/rows: 0


## Column Overview, Data Type Conversion, and Format Correction

### `employee_id`

In [394]:
print(df["employee_id"].value_counts())

employee_id
EMP1000    1
EMP1001    1
EMP1002    1
EMP1003    1
EMP1004    1
          ..
EMP2015    1
EMP2016    1
EMP2017    1
EMP2018    1
EMP2019    1
Name: count, Length: 1020, dtype: int64


### `first_name` and `last_name`

In [395]:
print(df["first_name"].unique())
print("")
print(df["last_name"].unique())

<StringArray>
['Bob', 'Alice', 'Eva', 'Frank', 'Charlie', 'David', 'Heidi', 'Grace']
Length: 8, dtype: str

<StringArray>
['Davis', 'Brown', 'Jones', 'Williams', 'Garcia', 'Johnson', 'Miller',
 'Smith']
Length: 8, dtype: str


In [396]:
df["full_name"] = df["first_name"] + " " + df["last_name"]

In [397]:
df.drop(columns=["first_name", "last_name"], inplace=True)

In [398]:
df.head(3)

,employee_id,age,department_region,status,join_date,salary,email,phone,performance_score,remote_work,full_name
0,EMP1000,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True,Bob Davis
1,EMP1001,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True,Bob Brown
2,EMP1002,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True,Alice Jones


### `email`

It seems like the email is the combination of first and last name ($8\times 8 = 64$).

In [399]:
print(df["email"].value_counts())

email
grace.brown@example.com      27
frank.smith@example.com      25
grace.smith@example.com      23
charlie.jones@example.com    22
frank.brown@example.com      22
                             ..
eva.johnson@example.com      10
heidi.davis@example.com      10
heidi.brown@example.com       9
alice.jones@example.com       7
david.jones@example.com       7
Name: count, Length: 64, dtype: int64


In [400]:
print("Number of emails that don't follow the correct email format:", (~df["email"].str.match(r"[\w\.]+@\w+\.\w+")).sum())

Number of emails that don't follow the correct email format: 0


### `age`

In [401]:
print(df["age"].value_counts())

age
40.0    210
25.0    206
30.0    205
35.0    188
Name: count, dtype: int64


In [402]:
print("NaN values inside column 'age':", df["age"].isna().sum())

NaN values inside column 'age': 211


Cleaning the 211 na values requires a bit of EDA to have a reasonable interpolation instead of filling in with median or such constant value. This basic EDA, for example, could be studying for patterns between the employee's name and age.

### `department_region`

In [403]:
print(np.sort(df["department_region"].unique()))

['Admin-California' 'Admin-Florida' 'Admin-Illinois' 'Admin-Nevada'
 'Admin-New York' 'Admin-Texas' 'Cloud Tech-California'
 'Cloud Tech-Florida' 'Cloud Tech-Illinois' 'Cloud Tech-Nevada'
 'Cloud Tech-New York' 'Cloud Tech-Texas' 'DevOps-California'
 'DevOps-Florida' 'DevOps-Illinois' 'DevOps-Nevada' 'DevOps-New York'
 'DevOps-Texas' 'Finance-California' 'Finance-Florida' 'Finance-Illinois'
 'Finance-Nevada' 'Finance-New York' 'Finance-Texas' 'HR-California'
 'HR-Florida' 'HR-Illinois' 'HR-Nevada' 'HR-New York' 'HR-Texas'
 'Sales-California' 'Sales-Florida' 'Sales-Illinois' 'Sales-Nevada'
 'Sales-New York' 'Sales-Texas']


In [404]:
df["department"] = df["department_region"].str.extract(r"([\w\s]+)(?=-)")
df["region"] = df["department_region"].str.extract(r"(?<=-)([\w\s]+)")

In [405]:
df.drop(columns="department_region", inplace=True)

In [406]:
df.head(3)

,employee_id,age,status,join_date,salary,email,phone,performance_score,remote_work,full_name,department,region
0,EMP1000,25.0,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True,Bob Davis,DevOps,California
1,EMP1001,NaN,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True,Bob Brown,Finance,Texas
2,EMP1002,NaN,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True,Alice Jones,Admin,Nevada


### `status`

In [407]:
print(df["status"].value_counts())

status
Pending     356
Active      352
Inactive    312
Name: count, dtype: int64


### `join_date`

In [408]:
df["join_date"] = pd.to_datetime(df["join_date"], errors="coerce", format="mixed")

In [409]:
print("NaN values inside the formatted column `join_date`:", df["join_date"].isna().sum())

NaN values inside the formatted column `join_date`: 0


In [410]:
print(df["join_date"].describe())

count                          1020
mean     2022-07-26 12:53:38.823529
min             2020-01-01 00:00:00
25%             2021-04-12 06:00:00
50%             2022-08-09 12:00:00
75%             2023-10-31 12:00:00
max             2024-12-29 00:00:00
Name: join_date, dtype: object


### `salary`

In [411]:
print(df["salary"].describe())

count       996.000000
mean      85155.056396
std       19873.727918
min       50047.320000
25%       68392.487500
50%       85547.870000
75%      100974.027500
max      119971.650000
Name: salary, dtype: float64


In [412]:
print("NaN values inside column `salary`:", df["salary"].isna().sum())

NaN values inside column `salary`: 24


Cleaning the 24 na values requires a bit of EDA to have a reasonable interpolation instead of filling in with mean or such constant value. This basic EDA, for example, could be studying for patterns among the employee's department region, age and salary.

### `phone`

In [413]:
df["phone"].describe()

count    1.020000e+03
mean    -4.942253e+09
std      2.817326e+09
min     -9.994973e+09
25%     -7.341992e+09
50%     -4.943997e+09
75%     -2.520391e+09
max     -3.896086e+06
Name: phone, dtype: float64

In [414]:
df["phone"] = df["phone"].abs().astype(str)

In [415]:
def fill_phone_with_zeros(phone: str):
    """Prepend zeros so that the phone number is a length of 10.
    
    For a given phone number of length 10 or greater, the number itself is returned back.
    """
    length = len(phone)
    zeros_count = 10 - length
    zeros_str = "0" * zeros_count
    return zeros_str + phone

In [416]:
df["phone"] = df["phone"].apply(lambda x: fill_phone_with_zeros(x))
df["phone"] = df["phone"].apply(lambda x: f"{x[:3]}-{x[3:6]}-{x[6:]}")

In [417]:
invalid_phone_mask = ~df["phone"].str.match(r"\d{3}-\d{3}-\d{4}")

print("Count of phone numbers that don't abide the general format:", df.loc[invalid_phone_mask, "phone"].shape[0])

Count of phone numbers that don't abide the general format: 0


### `performance_score`

In [418]:
print(df["performance_score"].value_counts())

performance_score
Good         270
Average      267
Excellent    267
Poor         216
Name: count, dtype: int64


### `remote_work`

In [419]:
print(df["remote_work"].value_counts())

remote_work
True     513
False    507
Name: count, dtype: int64


### Rearranging Columns

In [420]:
print(df.columns.tolist())

['employee_id', 'age', 'status', 'join_date', 'salary', 'email', 'phone', 'performance_score', 'remote_work', 'full_name', 'department', 'region']


In [421]:
column_order = ["employee_id", "full_name", "age", "email", "phone", "department", "region", "join_date", "salary", "status", "performance_score", "remote_work"]

df = df[column_order]

display(df.head(3))

,employee_id,full_name,age,email,phone,department,region,join_date,salary,status,performance_score,remote_work
0,EMP1000,Bob Davis,25.0,bob.davis@example.com,165-162-3197,DevOps,California,2021-04-02,59767.65,Active,Average,True
1,EMP1001,Bob Brown,NaN,bob.brown@example.com,189-847-1390,Finance,Texas,2020-07-10,65304.66,Active,Excellent,True
2,EMP1002,Alice Jones,NaN,alice.jones@example.com,559-636-3211,Admin,Nevada,2023-12-07,88145.90,Pending,Good,True


## Interpolation with Basic EDA

### `age` VS `department` x `region`

I assume employees in similar roles and region often have similar demographic averages like age.

In [422]:
df["temp_department_region"] = df["department"] + "-" + df["region"]

df["age"] = df.groupby(["temp_department_region"])["age"].transform(lambda x: x.fillna(x.median()))

In [423]:
df["age"] = df["age"].round().astype("Int64")

In [424]:
print("NaN values inside column `age`:", df["age"].isna().sum())

NaN values inside column `age`: 0


### `salary` VS `department` x `region`

In [425]:
df.head(5)

,employee_id,full_name,age,email,phone,department,region,join_date,salary,status,performance_score,remote_work,temp_department_region
0,EMP1000,Bob Davis,25,bob.davis@example.com,165-162-3197,DevOps,California,2021-04-02,59767.65,Active,Average,True,DevOps-California
1,EMP1001,Bob Brown,35,bob.brown@example.com,189-847-1390,Finance,Texas,2020-07-10,65304.66,Active,Excellent,True,Finance-Texas
2,EMP1002,Alice Jones,35,alice.jones@example.com,559-636-3211,Admin,Nevada,2023-12-07,88145.90,Pending,Good,True,Admin-Nevada
3,EMP1003,Eva Davis,25,eva.davis@example.com,347-649-0784,Admin,Nevada,2021-11-27,69450.99,Inactive,Good,True,Admin-Nevada
4,EMP1004,Frank Williams,25,frank.williams@example.com,158-673-4256,Cloud Tech,Florida,2022-01-05,109324.61,Active,Poor,False,Cloud Tech-Florida


Since performance is a volatile factor that doesn't reliably scale with salary, stick to the most stable demographic indicators: Department and Region.

In [426]:
score_order = ["Poor", "Average", "Good", "Excellent"]
df["performance_score"] = pd.Categorical(
    df["performance_score"],
    categories=score_order,
    ordered=True
)

display(df.groupby(["temp_department_region", "performance_score"], observed=True)["salary"].mean().head(20))

temp_department_region  performance_score
Admin-California        Poor                  88722.050000
                        Average               83949.028000
                        Good                  96891.746667
                        Excellent             84870.498182
Admin-Florida           Poor                  87487.512500
                        Average               85033.504286
                        Good                  85554.277500
                        Excellent             87049.061667
Admin-Illinois          Poor                  98242.520000
                        Average               88400.721000
                        Good                  86348.517143
                        Excellent            100094.225000
Admin-Nevada            Poor                  84689.985000
                        Average               72099.376667
                        Good                  81513.786667
                        Excellent             85220.983750
Admin-New York

You should almost always use median when dealing with salary or income data because salaries are highly susceptible to outliers.

In [427]:
df["salary"] = df.groupby("temp_department_region")["salary"].transform(lambda x: x.fillna(x.median()))

In [428]:
df["salary"] = df["salary"].round(2)

In [429]:
print("NaN values inside column `salary`:", df["salary"].isna().sum())

NaN values inside column `salary`: 0


In [430]:
df.drop(columns="temp_department_region", inplace=True)

## Exporting Final Cleaned Dataset

In [431]:
# I know that because of the uniqueness of the employee_id there won't any duplicates but for the sake of the culture, i still do this
print("Number of duplicated records/rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)

Number of duplicated records/rows: 0


In [432]:
display(df)

,employee_id,full_name,age,email,phone,department,region,join_date,salary,status,performance_score,remote_work
0,EMP1000,Bob Davis,25,bob.davis@example.com,165-162-3197,DevOps,California,2021-04-02,59767.65,Active,Average,True
1,EMP1001,Bob Brown,35,bob.brown@example.com,189-847-1390,Finance,Texas,2020-07-10,65304.66,Active,Excellent,True
2,EMP1002,Alice Jones,35,alice.jones@example.com,559-636-3211,Admin,Nevada,2023-12-07,88145.90,Pending,Good,True
3,EMP1003,Eva Davis,25,eva.davis@example.com,347-649-0784,Admin,Nevada,2021-11-27,69450.99,Inactive,Good,True
4,EMP1004,Frank Williams,25,frank.williams@example.com,158-673-4256,Cloud Tech,Florida,2022-01-05,109324.61,Active,Poor,False
...,...,...,...,...,...,...,...,...,...,...,...,...
1015,EMP2015,David Miller,30,david.miller@example.com,354-621-2759,HR,California,2023-08-19,85969.54,Active,Good,True
1016,EMP2016,David Johnson,30,david.johnson@example.com,250-826-1122,Cloud Tech,Texas,2021-11-07,100215.06,Inactive,Good,True
1017,EMP2017,Charlie Williams,40,charlie.williams@example.com,126-163-2487,Finance,New York,2023-10-04,114587.11,Active,Average,False
1018,EMP2018,Alice Garcia,30,alice.garcia@example.com,899-572-9892,HR,Florida,2024-12-16,71318.79,Inactive,Good,True


In [433]:
df.to_csv("data/clean.csv", date_format="%Y-%m-%d", index=False)
print("Saved successfully!")

Saved successfully!
